In [5]:
from google.colab import drive
yielda
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!unzip /content/drive/MyDrive/GEOAI/Train.zip -d /content/

Archive:  /content/drive/MyDrive/GEOAI/Train.zip
replace /content/Orenburg_training_samples.shx? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
  inflating: /content/Orenburg_training_samples.shx  
replace /content/Orenburg_training_samples.shp? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/Orenburg_training_samples.shp  
replace /content/Fergana_training_samples.shp? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/Fergana_training_samples.shp? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/Fergana_training_samples.shp? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/Fergana_training_samples.shp? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/Fergana_training_samples.shp  
replace /content/Fergana_training_samples.dbf? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/Fergana_training_samples.dbf  
replace /content/Fergana_training_sam

In [20]:
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.spatial import cKDTree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
import os

# Import TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Load and preprocess data (keep this the same)
def load_data():
    # Check for required shapefile components
    required_files = [
        "/content/Fergana_training_samples.shp",
        "/content/Fergana_training_samples.shx",
        "/content/Fergana_training_samples.dbf",
        "/content/Orenburg_training_samples.shp",
        "/content/Orenburg_training_samples.shx",
        "/content/Orenburg_training_samples.dbf"
    ]
    missing_files = [f for f in required_files if not os.path.exists(f)]
    if missing_files:
        print(f"Error: The following required shapefile components are missing: {missing_files}")
        raise FileNotFoundError(f"Missing shapefile components: {missing_files}")

    # Load Sentinel data
    s1 = pd.read_csv("/content/drive/MyDrive/GEOAI/Sentinel1.csv")
    s2 = pd.read_csv("/content/drive/MyDrive/GEOAI/Sentinel2.csv")
    s1['ID'] = s1['ID'].astype(str)
    s2['ID'] = s2['ID'].astype(str)

    # Load shapefiles
    fergana_gdf = gpd.read_file("/content/Fergana_training_samples.shp")
    orenburg_gdf = gpd.read_file("/content/Orenburg_training_samples.shp")

    # Validate geometries
    if fergana_gdf.geometry.isna().any():
        raise ValueError("Invalid geometries in Fergana shapefile")
    if orenburg_gdf.geometry.isna().any():
        raise ValueError("Invalid geometries in Orenburg shapefile")

    # Handle CRS for Fergana
    if fergana_gdf.crs is None:
        fergana_gdf.set_crs("EPSG:4326", inplace=True)
    elif fergana_gdf.crs != "EPSG:4326":
        fergana_gdf = fergana_gdf.to_crs("EPSG:4326")

    # Handle CRS for Orenburg
    if orenburg_gdf.crs is None:
        orenburg_gdf.set_crs("EPSG:4326", inplace=True)
    elif orenburg_gdf.crs != "EPSG:4326":
        orenburg_gdf = orenburg_gdf.to_crs("EPSG:4326")

    # Combine GeoDataFrames
    train_gdf = gpd.GeoDataFrame(pd.concat([fergana_gdf, orenburg_gdf], ignore_index=True))
    train_gdf['ID'] = train_gdf['ID'].astype(str)

    # Extract coordinates
    train_gdf['lon'] = train_gdf.geometry.x
    train_gdf['lat'] = train_gdf.geometry.y

    # Check Fergana coordinates
    fergana_coords = fergana_gdf.geometry.bounds
    if not ((fergana_coords['minx'].between(71, 73)).all() and
            (fergana_coords['miny'].between(40, 42)).all()):
        print("Warning: Fergana coordinates may be incorrect!")

    return train_gdf[['ID', 'Cropland', 'lat', 'lon']], s1, s2

# 2. Feature aggregation (keep this the same)
def aggregate_features(df, id_col="ID"):
    valid_bands = ['VH', 'VV', 'B2', 'B3', 'B4', 'B5', 'B6',
                   'B7', 'B8', 'B8A', 'B11', 'B12']
    agg_cols = [col for col in df.columns if col in valid_bands]

    if agg_cols:
        overall_agg = df[[id_col] + agg_cols].groupby(id_col).agg(['mean', 'std', 'min', 'max']).reset_index()
        overall_agg.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in overall_agg.columns]
    else:
        overall_agg = pd.DataFrame({id_col: df[id_col].unique()})

    return overall_agg.fillna(0)

# 3. Main training and prediction pipeline (modified)
def train_and_predict():
    # Load data
    train_gdf, s1, s2 = load_data()
    test_meta = pd.read_csv("/content/drive/MyDrive/GEOAI/Test.csv")
    test_meta['ID'] = test_meta['ID'].astype(str)

    # Validate Sentinel-2 coordinates
    if not (s2["translated_lon"].between(-180, 180).all() and s2["translated_lat"].between(-90, 90).all()):
        print("Warning: Sentinel-2 coordinates are out of valid range!")

    # Approximate cropland label from nearest point
    tree = cKDTree(train_gdf[["lat", "lon"]].values)
    s2_points = s2.groupby("ID")[["translated_lat", "translated_lon"]].mean().reset_index()
    s2_points['ID'] = s2_points['ID'].astype(str)
    dist, idx = tree.query(s2_points[["translated_lat", "translated_lon"]].values, k=1)
    print("Sample distances from cKDTree:", dist[:5])
    if (dist > 1).any():
        print("Warning: Some points are far from training data!")

    s2_points["label"] = train_gdf.iloc[idx.flatten()]['Cropland'].values
    s2_labels = s2_points[["ID", "label"]]

    # Aggregate features
    s1_feats = aggregate_features(s1.drop(columns=[col for col in s1.columns if 'date' in col.lower()]))
    s2_feats = aggregate_features(s2.drop(columns=[col for col in s2.columns if 'date' in col.lower() or 'cloud' in col.lower()]))

    # Merge and prepare training data
    train_df = s2_feats.merge(s1_feats, on="ID", how="outer").merge(s2_labels, on="ID", how="inner")
    train_df = train_df.dropna()
    X = train_df.drop(columns=["ID", "label"])
    y = train_df["label"]

    X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Define the neural network model
    model = keras.Sequential([
        layers.Input(shape=(X_train_scaled.shape[1],)),  # Input layer
        layers.Dense(16, activation='relu'),           # Hidden layer 1 with 16 neurons and ReLU activation
        layers.Dense(8, activation='relu'),           # Hidden layer 2 with 8 neurons and ReLU activation
        layers.Dense(1, activation='sigmoid')          # Output layer for binary classification with sigmoid activation
    ])

    # Compile the model
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # Train the model with 16 epochs
    history = model.fit(X_train_scaled, y_train,
                        epochs=15,
                        batch_size = 4,
                        validation_data=(X_val_scaled, y_val),
                        verbose=1)

    print("✅ Model training completed")

    # Evaluate on validation set
    val_loss, val_acc = model.evaluate(X_val_scaled, y_val, verbose=0)
    print(f"✅ Validation Accuracy: {val_acc:.4f}")

    # Prepare test set
    test_ids = test_meta["ID"].unique()
    s1_test = s1[s1["ID"].isin(test_ids)].drop(columns=[col for col in s1.columns if 'date' in col.lower()])
    s2_test = s2[s2["ID"].isin(test_ids)].drop(columns=[col for col in s2.columns if 'date' in col.lower() or 'cloud' in col.lower()])

    s1_test_feats = aggregate_features(s1_test)
    s2_test_feats = aggregate_features(s2_test)
    test_df = s2_test_feats.merge(s1_test_feats, on="ID", how="outer").fillna(0)

    X_test = test_df.drop(columns=["ID"])

    # Ensure feature consistency
    for col in X.columns:
        if col not in X_test.columns:
            X_test[col] = 0
    X_test = X_test[X.columns]

    X_test_scaled = scaler.transform(X_test)

    # Predict
    test_preds_proba = model.predict(X_test_scaled)
    test_preds = (test_preds_proba > 0.5).astype(int).flatten()

    # Save submission
    submission = pd.DataFrame({"ID": test_df["ID"], "Target": test_preds})
    submission.to_csv("submission.csv", index=False)
    print("📤 submission.csv saved")

    return submission

# 4. Run pipeline
if __name__ == "__main__":
    submission = train_and_predict()

Sample distances from cKDTree: [0.0250008  0.06443743 0.02575809 0.05825472 0.02940408]
Epoch 1/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4411 - loss: 0.8307 - val_accuracy: 0.6250 - val_loss: 0.6488
Epoch 2/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6501 - loss: 0.6329 - val_accuracy: 0.6417 - val_loss: 0.6015
Epoch 3/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6656 - loss: 0.6051 - val_accuracy: 0.6667 - val_loss: 0.5957
Epoch 4/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6990 - loss: 0.5975 - val_accuracy: 0.6250 - val_loss: 0.6091
Epoch 5/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6794 - loss: 0.5625 - val_accuracy: 0.6583 - val_loss: 0.5881
Epoch 6/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6765 - loss: 0.5714 - val_accuracy: 0.6417 - val_loss: 0.5853
Epoch 7/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6712 - loss: 0.6062 - val_accuracy: 0.6250 - val_loss: 0.5880
Epoch 8/15
120/1